In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

# Churn Data
Imaginary data from a ride-sharing app usage in a set of imaginary cities.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
ride = pd.read_csv('./churn_data.csv')
ride.head()

In [ ]:
ride.shape

Use first month's worth of data to predict whether the app's users will remain users of the service at the 5 month mark

In [ ]:
X = ride.drop('month_5_still_here', axis=1)
y = ride['month_5_still_here']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=123)

## XGBoost Classifier (Logistic)

In [ ]:
# initiate xgbclassifier
xg_cl = xgb.XGBClassifier(objective='binary:logistic',
                          n_estimators=10,
                          seed=123)

In [ ]:
# fit and predict
xg_cl.fit(X_train, y_train)

preds = xg_cl.predict(X_test)

acc = accuracy_score(y_test, preds)
print(f'test acc: {acc: .3f}')

## Cross-validation using the XGBoost API

In [ ]:
# first, create a DMatrix from X and y
churn_dmatrix = xgb.DMatrix(data=X, label=y)

# craete parameter dictionary
params = {'objective': 'reg:logistic',
          'max_depth': 3}

In [ ]:
# perform cv
cv_results = xgb.cv(dtrain=churn_dmatrix,
                    params=params,
                    nfold=3,
                    num_boost_round=5, # no of trees
                    metrics='error',
                    as_pandas=True,
                    seed=123)
cv_results

In [ ]:
print(f'test acc: {1-cv_results.iloc[-1, 2]}')

In [ ]:
# using auc metric
# perform cv
cv_results = xgb.cv(dtrain=churn_dmatrix,
                    params=params,
                    nfold=3,
                    num_boost_round=5, # no of trees
                    metrics='auc',
                    as_pandas=True,
                    seed=123)
cv_results

# `sklearn` Breast Cancer Dataset

In [ ]:
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier

In [ ]:
breast = datasets.load_breast_cancer()
breast.keys()

In [ ]:
X = breast['data']
y = breast['target']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=123)

## Decision Tree Classifier

In [ ]:
# initiate tree classifier, fit, and eval
dt_clf_4 = DecisionTreeClassifier(max_depth=4)

dt_clf_4.fit(X_train, y_train)

y_pred = dt_clf_4.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'test acc: {acc: .3f}')

# Ames, Iowa Housing Prices Dataset

In [ ]:
pd.options.display.max_columns = None
pd.options.display.max_rows = None

In [ ]:
ames = pd.read_csv('./ames.csv')
ames.head()

In [ ]:
ames.info()

In [ ]:
ames_ = ames.copy()
ames_.fillna(0, inplace=True) #as they did
ames_ = pd.get_dummies(ames_, drop_first=True)
ames_.head()

In [ ]:
X = ames_.drop('SalePrice', axis=1)
y = ames_['SalePrice']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=123)

## XGBRegressor
with `tree` as base learners and using the `sklearn`'s API

In [ ]:
from sklearn.metrics import mean_squared_error as MSE

In [ ]:
# initiate regressor, fit, and predict
xg_reg = xgb.XGBRegressor(objective='reg:linear',
                          n_estimators=10,
                          booster='gbtree',
                          seed=123)

xg_reg.fit(X_train, y_train)

y_pred = xg_reg.predict(X_test)

# compute rmse
rmse = np.sqrt(MSE(y_test, y_pred))
print(f'RMSE: {rmse}')

## XGBRegressor
with `linear` as base learner and using `XGBoost`'s own learning API

In [ ]:
# first, convert training and test sets into DMatrices
DM_train = xgb.DMatrix(data=X_train, label=y_train)
DM_test = xgb.DMatrix(data=X_test, label=y_test)

# create parameter dict
params = {'booster': 'gblinear',
          'objective': 'reg:linear'}

In [ ]:
# train, predict, eval
xg_reg = xgb.train(dtrain=DM_train,
                   params=params,
                   num_boost_round=5)

y_pred = xg_reg.predict(DM_test)

rmse = np.sqrt(MSE(y_test, y_pred))
print(f'RMSE: {rmse}')

## Cross-validation

In [ ]:
# create DMatrix
housing_dmatrix = xgb.DMatrix(data=X, label=y)

# create params
params = {'objective': 'reg:linear',
          'max_depth': 4}

In [ ]:
# perform cv
cv_results = xgb.cv(dtrain=housing_dmatrix,
                    params=params,
                    nfold=4,
                    num_boost_round=5,
                    metrics='rmse',
                    as_pandas=True,
                    seed=123)
cv_results

In [ ]:
# perform cv with mae as metric
cv_results = xgb.cv(dtrain=housing_dmatrix,
                    params=params,
                    nfold=4,
                    num_boost_round=5,
                    metrics='mae',
                    as_pandas=True,
                    seed=123)
cv_results

## Using regularization in `XGBoost` (L2)

In [ ]:
# lambda (for L2 reg) values
reg_params = [1, 10, 100]

# create initial param dict
params = {'objective': 'reg:linear',
          'max_depth': 3}

In [ ]:
# create empty list for storing rmse as a fxn of l2 complexity
rmses_l2 = []

# iterate over reg_params:
for reg in reg_params:
    # update l2 strength
    params['lambda'] = reg
    
    # pass updated dict into cv
    cv_results_rmse = xgb.cv(dtrain=housing_dmatrix,
                             params=params,
                             nfold=2,
                             num_boost_round=5,
                             metrics='rmse',
                             as_pandas=True,
                             seed=123)
    
    # append best rmse (final) to list
    rmses_l2.append(cv_results_rmse['test-rmse-mean'].tail(1).values[0])

# look at best rmse per l2 param
print('Best rmse as a fxn of l2:')
pd.DataFrame(list(zip(reg_params, rmses_l2)),
             columns=['l2', 'rmse'])

## Visualizing individual `XGBoost` trees

In [ ]:
# create param dict
params = {'objective': 'reg:linear',
          'max_depth': 2}

In [ ]:
# train model
xg_reg = xgb.train(params=params,
                   dtrain=housing_dmatrix,
                   num_boost_round=10)

In [ ]:
# plot trees
fig, (ax1, ax2, ax3) = plt.subplots(ncols=3, nrows=1, figsize=(24, 8))

xgb.plot_tree(xg_reg, num_trees=0, ax=ax1)
xgb.plot_tree(xg_reg, num_trees=4, ax=ax2)
xgb.plot_tree(xg_reg, num_trees=9, rankdir='LR', ax=ax3)

ax1.set_title('Tree 0')
ax2.set_title('Tree 5')
ax3.set_title('Tree 10')

plt.show()

If you encounter an error with graphviz, do:
```
conda install graphviz
conda install python-graphviz
conda install pydot
```

## Visualizing feature importances

In [ ]:
# create param dict
params = {'objective': 'reg:linear',
          'max_depth': 4}

In [ ]:
# train model
xg_reg = xgb.train(params=params,
                   dtrain=housing_dmatrix,
                   num_boost_round=10)

In [ ]:
xgb.plot_importance(xg_reg)
plt.show()

## Tuning number of boosting rounds

In [ ]:
# create param dict for each tree
params = {'objective': 'reg:linear',
          'max_depth': 3}

# list of number of boosting rounds
num_rounds = [5, 10, 15]

# empty list to store final round rmse per XGBoost model
final_rmse_per_round = []

In [ ]:
# iterate over num_rounds
for current in num_rounds:
    #perform cv
    cv_results = xgb.cv(dtrain=housing_dmatrix,
                        params=params,
                        nfold=3,
                        num_boost_round=current,
                        metrics='rmse',
                        as_pandas=True,
                        seed=123)
    #append final round RMSE
    final_rmse_per_round.append(cv_results['test-rmse-mean'].tail().values[-1])

In [ ]:
# show results
num_rounds_rmses = list(zip(num_rounds, final_rmse_per_round))
pd.DataFrame(num_rounds_rmses, columns=['num_boosting_rounds', 'rmse'])

## Automated boosting round selection using `early_stopping_rounds`

In [ ]:
# create param dict for each tree
params = {'objective': 'reg:linear',
          'max_depth': 4}

In [ ]:
# perform cv with early stopping
cv_results = xgb.cv(params=params,
                    dtrain=housing_dmatrix,
                    nfold=3,
                    num_boost_round=50,
                    early_stopping_rounds=10,
                    metrics='rmse',
                    as_pandas=True,
                    seed=123)
cv_results

## Tuning `eta`, the learning rate

In [ ]:
# create param dict for each tree
params = {'objective': 'reg:linear',
          'max_depth': 3}

# list of eta values and empty list for storing final round rmse
eta_vals = [0.001, 0.01, 0.1]
best_rmse = []

In [ ]:
# iterate over etas
for curr in eta_vals:
    params['eta'] = curr
    
    #perform cv
    cv_results = xgb.cv(params=params,
                        dtrain=housing_dmatrix,
                        nfold=3,
                        num_boost_round=10,
                        early_stopping_rounds=5,
                        metrics='rmse',
                        as_pandas=True,
                        seed=123)
    
    #append final round rmse
    best_rmse.append(cv_results['test-rmse-mean'].tail().values[-1])

In [ ]:
# print results
pd.DataFrame(list(zip(eta_vals, best_rmse)),
             columns=['eta', 'best_rmse'])

## Tuning `max_depth`

In [ ]:
params = {'objective': 'reg:linear'}

max_depths = [2, 5, 10, 20]
best_rmse = []

In [ ]:
for curr in max_depths:
    params['max_depth'] = curr
    
    cv_results = xgb.cv(params=params,
                        dtrain=housing_dmatrix,
                        nfold=2,
                        num_boost_round=10,
                        early_stopping_rounds=5,
                        metrics='rmse',
                        as_pandas=True,
                        seed=123)
    
    best_rmse.append(cv_results['test-rmse-mean'].tail().values[-1])

In [ ]:
pd.DataFrame(list(zip(max_depths, best_rmse)),
            columns=['max_depth', 'best_rmse'])

## Tuning `colsample_bytree`

In [ ]:
params={"objective":"reg:linear",
        "max_depth":3}

colsample_bytree_vals = [0.1, 0.5, 0.8, 1]
best_rmse = []

In [ ]:
for curr in colsample_bytree_vals:
    params['colsample_bytree'] = curr
    
    cv_results = xgb.cv(params=params,
                        dtrain=housing_dmatrix,
                        nfold=2,
                        num_boost_round=10,
                        early_stopping_rounds=5,
                        metrics='rmse',
                        as_pandas=True,
                        seed=123)
    
    best_rmse.append(cv_results['test-rmse-mean'].tail().values[-1])

In [ ]:
pd.DataFrame(list(zip(colsample_bytree_vals, best_rmse)),
            columns=['colsample_bytree', 'best_rmse'])

## Grid Search with `XGBoost`

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
# create parameter grid
gbm_param_grid = {'colsample_bytree': [0.3, 0.7],
                  'n_estimators': [50],
                  'max_depth': [2, 5]}

In [ ]:
# initiate regressor
gbm = xgb.XGBRegressor()

In [ ]:
# initiate grid search object
grid_mse = GridSearchCV(estimator=gbm,
                        param_grid=gbm_param_grid,
                        scoring='neg_mean_squared_error',
                        cv=4,
                        verbose=1)

In [ ]:
grid_mse.fit(X, y)

In [ ]:
print(f'Best params: {grid_mse.best_params_}')
print(f'Lowest RMSE: {np.sqrt(np.abs(grid_mse.best_score_))}')

## Random Search with `XGBoost`

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
gbm_param_grid = {'n_estimators': [25],
                  'max_depth': range(2, 12)}

In [ ]:
gbm = xgb.XGBRegressor(n_estimators=10)

In [ ]:
random_mse = RandomizedSearchCV(estimator=gbm,
                                param_distributions=gbm_param_grid,
                                scoring='neg_mean_squared_error',
                                n_iter=5,
                                cv=4, 
                                verbose=1)

In [ ]:
random_mse.fit(X, y)

In [ ]:
print(f'Best params: {random_mse.best_params_}')
print(f'Lowest RMSE: {np.sqrt(np.abs(random_mse.best_score_))}')